### Dataset and Task Metadata

In [5]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="audiology_diagnosis",
    dataset_year="1987",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5TP4R",
    download_description="""
We get the data from UCI.
wget https://archive.ics.uci.edu/static/public/8/audiology+standardized.zip && unzip audiology+standardized.zip && rm audiology+standardized.zip audiology.standardized.names && mkdir -p local-data-warehouse/audiology_diagnosis && mv audiology.standardized.data audiology.standardized.test local-data-warehouse/audiology_diagnosis/
""",
    # References
    academic_reference_bibtex="""@incollection{bareiss1990protos,
  title={Protos: An exemplar-based learning apprentice},
  author={Bareiss, E Ray and Porter, Bruce W and Wier, Craig C},
  booktitle={Machine learning},
  pages={112--127},
  year={1990},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="bareiss1990protos",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI and merge train and test data.

- We drop the ID column as it is uninformative here.
- The target label contains groups of labels with specifications. We merge labels into groups that represent general a diagnosis. We create 3 labels: "normal", "cochlear", "other". There is likely a much better way to partition these labels, but this is most reasonable partitioning for an acutal task, going from names and my limited domain knowledge about the various diagnoses.
- We drop duplicated rows as they might introduce too much information leakage for such small data and are likely not natural but rather an artifact of the limited number of features.
- We remove one constant column (history_fullness).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="diagnosis",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="diagnosis",
)

## Preprocessing

In [6]:
import pandas as pd
import numpy as np
columns = [
    "age_gt_60","air","airBoneGap","ar_c","ar_u","bone","boneAbnormal","bser","history_buzzing","history_dizziness","history_fluctuating","history_fullness","history_heredity","history_nausea","history_noise","history_recruitment","history_ringing","history_roaring","history_vomiting","late_wave_poor","m_at_2k","m_cond_lt_1k","m_gt_1k","m_m_gt_2k","m_m_sn","m_m_sn_gt_1k","m_m_sn_gt_2k","m_m_sn_gt_500","m_p_sn_gt_2k","m_s_gt_500","m_s_sn","m_s_sn_gt_1k","m_s_sn_gt_2k","m_s_sn_gt_3k","m_s_sn_gt_4k","m_sn_2_3k","m_sn_gt_1k","m_sn_gt_2k","m_sn_gt_3k","m_sn_gt_4k","m_sn_gt_500","m_sn_gt_6k","m_sn_lt_1k","m_sn_lt_2k","m_sn_lt_3k","middle_wave_poor","mod_gt_4k","mod_mixed","mod_s_mixed","mod_s_sn_gt_500","mod_sn","mod_sn_gt_1k","mod_sn_gt_2k","mod_sn_gt_3k","mod_sn_gt_4k","mod_sn_gt_500","notch_4k","notch_at_4k","o_ar_c","o_ar_u","s_sn_gt_1k","s_sn_gt_2k","s_sn_gt_4k","speech","static_normal","tymp","viith_nerve_signs","wave_V_delayed","waveform_ItoV_prolonged","indentifier", "diagnosis"
]
df = pd.read_csv(dataset_mold.path / "audiology.standardized (1).data", header=None, names=columns, na_values="?")
df = pd.concat([df, pd.read_csv(dataset_mold.path / "audiology.standardized (1).test", header=None, names=columns, na_values="?")], ignore_index=True)
print("Loaded data shape:", df.shape)

as_cat_type = list(df)
df[as_cat_type] = df[as_cat_type].astype("category")


cochlear_classes = [
    "cochlear_age_and_noise",
    "cochlear_age_plus_poss_menieres",
    "cochlear_noise_and_heredity",
    "cochlear_poss_noise",
    "cochlear_unknown",
    "possible_menieres",
    "mixed_cochlear_age_fixation",
    "mixed_cochlear_age_otitis_media",
    "mixed_cochlear_age_s_om",
    "mixed_cochlear_unk_discontinuity",
    "mixed_cochlear_unk_fixation",
    "mixed_cochlear_unk_ser_om",
    "cochlear_age",
    "mixed_poss_noise_om",
]
normal_classes = ["normal_ear"]
other_classes = [
    "acoustic_neuroma",
    "bells_palsy",
    "conductive_discontinuity",
    "conductive_fixation",
    "mixed_poss_central_om",
    "otitis_media",
    "poss_central",
    "possible_brainstem_disorder",
    "retrocochlear_unknown",
]
def map_to_diagnosis(x):
    if x in cochlear_classes:
        return "cochlear"
    elif x in normal_classes:
        return "normal"
    elif x in other_classes:
        return "other"
    else:
        raise ValueError(f"Unknown diagnosis class: {x}")
df["diagnosis"] = df["diagnosis"].apply(map_to_diagnosis).astype("category")
df = df.drop(columns=[
    "indentifier",
    "history_fullness", # constant column
])
df = df.drop_duplicates()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (226, 71)


## Data Checks

In [7]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 199
Columns: 69
Use sampling: False (sample size: 199)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['speech', 'tymp', 'air', 'bone', 'o_ar_u', 'ar_u', 'ar_c', 'o_ar_c', 'm_sn_gt_3k', 'm_sn_gt_4k']
Rows remaining as candidates after top-10 filter: 75 (of 199)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [8]:
# Sample Rows
df_head

,age_gt_60,air,airBoneGap,ar_c,ar_u,bone,boneAbnormal,bser,history_buzzing,history_dizziness,history_fluctuating,history_heredity,history_nausea,history_noise,history_recruitment,history_ringing,history_roaring,history_vomiting,late_wave_poor,m_at_2k,m_cond_lt_1k,m_gt_1k,m_m_gt_2k,m_m_sn,m_m_sn_gt_1k,m_m_sn_gt_2k,m_m_sn_gt_500,m_p_sn_gt_2k,m_s_gt_500,m_s_sn,m_s_sn_gt_1k,m_s_sn_gt_2k,m_s_sn_gt_3k,m_s_sn_gt_4k,m_sn_2_3k,m_sn_gt_1k,m_sn_gt_2k,m_sn_gt_3k,m_sn_gt_4k,m_sn_gt_500,m_sn_gt_6k,m_sn_lt_1k,m_sn_lt_2k,m_sn_lt_3k,middle_wave_poor,mod_gt_4k,mod_mixed,mod_s_mixed,mod_s_sn_gt_500,mod_sn,mod_sn_gt_1k,mod_sn_gt_2k,mod_sn_gt_3k,mod_sn_gt_4k,mod_sn_gt_500,notch_4k,notch_at_4k,o_ar_c,o_ar_u,s_sn_gt_1k,s_sn_gt_2k,s_sn_gt_4k,speech,static_normal,tymp,viith_nerve_signs,wave_V_delayed,waveform_ItoV_prolonged,diagnosis
0,f,mild,f,elevated,normal,mild,t,NaN,f,f,f,f,f,t,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,elevated,normal,f,f,f,poor,t,c,f,f,f,cochlear
1,t,mild,f,normal,elevated,mild,t,NaN,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,normal,elevated,f,f,f,good,t,a,f,f,f,cochlear
2,t,normal,f,absent,absent,normal,f,NaN,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,f,f,f,f,NaN,absent,f,f,f,normal,t,c,f,f,f,cochlear
3,t,mild,f,normal,normal,unmeasured,f,NaN,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,t,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,normal,normal,f,f,f,good,t,a,f,f,f,cochlear
4,f,normal,f,normal,elevated,NaN,f,NaN,f,f,f,t,f,t,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,f,normal,normal,f,t,f,good,t,a,f,f,f,cochlear


In [9]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,bser,category,196.0,98.49,2.0,"normal, degraded"
1,bone,category,65.0,32.66,4.0,"mild, unmeasured, normal, moderate"
2,o_ar_c,category,5.0,2.51,3.0,"normal, absent, elevated"
3,speech,category,5.0,2.51,6.0,"normal, good, very_good, very_poor, poor, unmeasured"
4,ar_c,category,4.0,2.01,3.0,"normal, absent, elevated"
5,ar_u,category,3.0,1.51,3.0,"normal, absent, elevated"
6,o_ar_u,category,2.0,1.01,3.0,"normal, absent, elevated"
7,age_gt_60,category,0.0,0.00,2.0,"f, t"
8,air,category,0.0,0.00,5.0,"mild, normal, moderate, severe, profound"
9,airBoneGap,category,0.0,0.00,2.0,"f, t"


In [10]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [11]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                  rank                          
age_gt_60               1              f    121  60.80
                        2              t     78  39.20
air                     1           mild     98  49.25
                        2         normal     72  36.18
                        3       moderate     21  10.55
                        4         severe      7   3.52
                        5       profound      1   0.50
airBoneGap              1              f    176  88.44
                        2              t     23  11.56
ar_c                    1         normal    112  56.28
                        2         absent     50  25.13
                        3       elevated     33  16.58
                        4           <NA>      4   2.01
ar_u                    1         normal    118  59.30
                        2         absent     43  21.61
                        3       elevated     35  17.59
                        4           <NA>      3   1.51
bone                    1           <NA>     65  32.66
                        2           mild     56  28.14
                        3     unmeasured     39  19.60
                        4         normal     35  17.59
                        5       moderate      4   2.01
boneAbnormal            1              f    155  77.89
                        2              t     44  22.11
bser                    1           <NA>    196  98.49
                        2         normal      2   1.01
                        3       degraded      1   0.50
diagnosis               1       cochlear    162  81.41
                        2         normal     19   9.55
                        3          other     18   9.05
history_buzzing         1              f    198  99.50
                        2              t      1   0.50
history_dizziness       1              f    180  90.45
                        2              t     19   9.55
history_fluctuating     1              f    192  96.48
                        2              t      7   3.52
history_heredity        1              f    197  98.99
                        2              t      2   1.01
history_nausea          1              f    189  94.97
                        2              t     10   5.03
history_noise           1              f    141  70.85
                        2              t     58  29.15
history_recruitment     1              f    197  98.99
                        2              t      2   1.01
history_ringing         1              f    190  95.48
                        2              t      9   4.52
history_roaring         1              f    189  94.97
                        2              t     10   5.03
history_vomiting        1              f    193  96.98
                        2              t      6   3.02
late_wave_poor          1              f    197  98.99
                        2              t      2   1.01
m_at_2k                 1              f    198  99.50
                        2              t      1   0.50
m_cond_lt_1k            1              f    198  99.50
                        2              t      1   0.50
m_gt_1k                 1              f    198  99.50
                        2              t      1   0.50
m_m_gt_2k               1              f    197  98.99
                        2              t      2   1.01
m_m_sn                  1              f    191  95.98
                        2              t      8   4.02
m_m_sn_gt_1k            1              f    194  97.49
                        2              t      5   2.51
m_m_sn_gt_2k            1              f    194  97.49
                        2              t      5   2.51
m_m_sn_gt_500           1              f    198  99.50
                        2              t      1   0.50
m_p_sn_gt_2k            1              f    197  98.99
                        2              t      2   1.01
m_s_gt_500              1              f    198  99.50
                    

In [12]:
# Target Distribution
target_df

,count,pct
diagnosis,,
cochlear,162,81.41
normal,19,9.55
other,18,9.05


## Task Curation

In [13]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [14]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [15]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to audiology_diagnosis/019d9cbe-7fc4-788e-bd47-f4023e92a32a
019d9cbe-7fc4-788e-bd47-f4023e92a32a
e220446ab6de9186af9cf5401b02e9d9cda60e22a2707504abd8f6e73d98897a
